#### Day1. 로지스틱 회귀(와인 데이터셋)

와인(wine) 데이터를 사용해 로지스틱 회귀를 수행합니다.

In [2]:
import pandas as pd
pd.set_option('display.width', 120)
path = "https://raw.githubusercontent.com/Soyoung-Yoon/data_01/main/"
df = pd.read_csv(path + "wine01.csv")
print(df.head(3))

   alcohol  malic_acid   ash  alcalinity_of_ash  magnesium  total_phenols  flavanoids  nonflavanoid_phenols  \
0    14.23        1.71  2.43               15.6      127.0           2.80        3.06                  0.28   
1    13.20        1.78  2.14               11.2      100.0           2.65        2.76                  0.26   
2    13.16        2.36  2.67               18.6      101.0           2.80        3.24                  0.30   

   proanthocyanins  color_intensity   hue  od280/od315_of_diluted_wines  proline  wine_variety  
0             2.29             5.64  1.04                          3.92   1065.0             0  
1             1.28             4.38  1.05                          3.40   1050.0             0  
2             2.81             5.68  1.03                          3.17   1185.0             0  


In [3]:
# 1-1) 종속변수는 'wine_variety' 입니다.
# 범주의 종류 및 개수를 확인해 봅니다.
print(df['wine_variety'].value_counts())

wine_variety
1    71
0    59
Name: count, dtype: int64


다음과 같은 로지스틱회귀 모형을 사용한 분류모델을 만들고 결과를 확인합니다.
- wine01.csv 데이터를 사용합니다.
- 모델 생성시 상수항(=절편)을 포함하도록 하며, 규제는 사용하지 않습니다.
- 종속변수 : wine_variety
- 독립변수 : alcohol, color_intensity, proline, flavanoids, malic_acid

In [4]:
# 1-2) GLM.from_formula() 를 사용해 분석하려고 합니다.
# formula를 작성하고, 로지스틱 회귀모형을 생성합니다.
from statsmodels.api import GLM, families
formula = "wine_variety ~ alcohol + color_intensity + proline + flavanoids + malic_acid"
model = GLM.from_formula(formula, df, family=families.Binomial()).fit()
print(model.summary())

                 Generalized Linear Model Regression Results                  
Dep. Variable:           wine_variety   No. Observations:                  130
Model:                            GLM   Df Residuals:                      124
Model Family:                Binomial   Df Model:                            5
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -3.2057
Date:                Mon, 10 Nov 2025   Deviance:                       6.4113
Time:                        20:37:06   Pearson chi2:                     5.93
No. Iterations:                    12   Pseudo R-squ. (CS):             0.7351
Covariance Type:            nonrobust                                         
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         178.6036    134.399     

In [6]:
#1-3) 모델의 로그-우도를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model.llf, 3))

-3.206


In [7]:
#1-4) 잔차이탈도(Deviance)를 반올림하여 소수점 아래 3자리까지 출력합니다.
print(round(model.deviance, 3))

6.411


In [13]:
#1-5) 'proline'을 독립변수로 하였을 때의 오즈비(Odds Ratio)는?
# 반올림하여 소수점 아래 3자리까지 출력합니다.
import numpy as np
odds_ratio = np.exp(model.params['proline'])
print(round(odds_ratio, 3))

0.975


In [16]:
#1-6) 'proline'가 3증가하면 오즈는 몇 % 감소 또는 증가하는가?
#(단, 감소율 또는 증가율은 반올림하여 소수점아래 2자리까지 표시합니다.)
# 감소율 = (1 - odds_ratio) * 100
# 증가율 = (odds_ratio - 1) * 100
import numpy as np
odds_ratio = np.exp(model.params['proline'] * 3)
rate = (1 - odds_ratio) * 100
print(round(rate, 2))


7.45


In [19]:
# 1-7) 아래의 sample을 사용하여 P(Y=1)에 대한 확률을 구하고,
# 반올림하여 소수점 아래 3자리까지 출력하세요.
# sample => alcohol : 13.5, color_intensity: 5.0, proline : 450, flavanoids : 2.8, malic_acid : 1.8
sample = {
    'alcohol': [13.5],
    'color_intensity': [5.0],
    'proline': [450],
    'flavanoids': [2.8],
    'malic_acid': [1.8]
}
temp = pd.DataFrame(sample)
p_y1 = round(model.predict(temp), 3)
print(p_y1)


0    0.659
dtype: float64


In [20]:
# 1-8) 위 샘플에 대한 odds 를 구하고,
# 반올림하여 소수점 아래 4자리까지 출력하세요.
odds = p_y1 / (1 - p_y1)
print(round(odds, 4))

0    1.9326
dtype: float64


In [23]:
#1-9) 유의수준 5%하에서, 유의성이 낮은 변수의 개수는 몇 개인가?
print(sum(model.pvalues[1:] > 0.05))

5


In [29]:
#1-10) 아래 샘플에 대한 P(Y=1)에 대한 95% 신뢰구간의 상한은?
# 반올림하여 소수점 아래 4자리까지 출력합니다.
# sample => alcohol : 13.5, color_intensity: 5.0, proline : 850, flavanoids : 2.8, malic_acid : 1.8
sample = {
    'alcohol': [13.5],
    'color_intensity': [5.0],
    'proline': [850],
    'flavanoids': [2.8],
    'malic_acid': [1.8]
}
temp = pd.DataFrame(sample)
result = model.get_prediction(temp)
print(round(result.conf_int(alpha=0.05)[0][1], 4))

0.9904


In [32]:
#1-11) 정확도를 구해 반올림하여 소수점 아래 3자리까지 출력합니다.
from sklearn.metrics import accuracy_score
y_true = df['wine_variety']
y_pred = model.predict(df).round().astype('int32')
acc = accuracy_score(y_true, y_pred)
print(round(acc, 3))

0.985


In [33]:
#1-12) f1_score를 구해 반올림하여 소수점 아래 3자리까지 출력합니다.
from sklearn.metrics import f1_score
y_true = df['wine_variety']
y_pred = model.predict(df).round().astype('int32')
acc = f1_score(y_true, y_pred)
print(round(acc, 3))

0.986
